The alternative to do not have the error showed before is:
```mermaid
flowchart TD
    U([User: How do I run Olama?])
    L1[LLM: I'll search for 'Olama']
    S1[search - Olama - no useful results]
    L2[LLM: Hmm, no results. Maybe a typo for 'Ollama'?]
    S2[search - Ollama - found results!]
    A([LLM: Here's how to run Ollama locally...])

    U --> L1 --> S1 --> L2 --> S2 --> A
```

In [1]:
from langchain_ollama import ChatOllama
from src.retrieve_documents import retrieve_documents
from src.searching_tool import fit_docs

documents = retrieve_documents()

index = fit_docs(documents)

Number of documents retrieved: 1346


In [2]:
# retrieve_llm = ChatOllama(model="gemma4:e2b")
retrieve_llm = ChatOllama(model="batiia-gemma-mini:latest")


In [3]:
from langchain_core.tools import tool

In [4]:
from typing import List, Dict

In [5]:
## we define a tool to use it on the agentic rag
@tool
def search(query: str) -> List[Dict]:
    """Search the FAQ database for entries matching the given query.
    Args:
        query: Search query text to look up in the course FAQ.
    Returns:
        List[Dict]: List of documents that are relevant to the query
    """
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
import json
print(json.dumps(search.args_schema.model_json_schema(), indent=2))

{
  "description": "Search the FAQ database for entries matching the given query.\nArgs:\n    query: Search query text to look up in the course FAQ.\nReturns:\n    List[Dict]: List of documents that are relevant to the query",
  "properties": {
    "query": {
      "title": "Query",
      "type": "string"
    }
  },
  "required": [
    "query"
  ],
  "title": "search",
  "type": "object"
}


In [7]:
tools = [search]
llm_with_tools = retrieve_llm.bind_tools(tools, tool_choice=True)

In [33]:
from langchain_core.messages import HumanMessage, ToolMessage

In [9]:
messages = [
    HumanMessage(content="I just discovered the course. Can I join it?")
]
response = llm_with_tools.invoke(messages)

In [ ]:
call = response.tool_calls[0]
args = call["args"]

results = search.invoke(args)
result_json = json.dumps(results, indent=2)

In [36]:
function_call_output = ToolMessage(
    content=result_json, 
    tool_call_id=call["id"]
)

In [38]:
messages.append(response)
messages.append(function_call_output)

In [39]:
messages

[HumanMessage(content='I just discovered the course. Can I join it?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'batiia-gemma-mini:latest', 'created_at': '2026-06-18T05:44:00.030674113Z', 'done': True, 'done_reason': 'stop', 'total_duration': 129268075337, 'load_duration': 119534463915, 'prompt_eval_count': 119, 'prompt_eval_duration': 987230543, 'eval_count': 272, 'eval_duration': 8294513644, 'logprobs': None, 'model_name': 'batiia-gemma-mini:latest', 'model_provider': 'ollama'}, id='lc_run--019ed940-27c1-7c03-8e4b-1716e48275e3-0', tool_calls=[{'name': 'search', 'args': {'query': 'how to join the course'}, 'id': 'b0e6855b-fc66-4578-bdee-81307670f3b4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 272, 'total_tokens': 391}),
 ToolMessage(content='[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions"

In [40]:
response = llm_with_tools.invoke(messages)

In [42]:
response

AIMessage(content='Yes, you can join the course!\n\nHowever, please note this important detail: **If you want to receive a certificate, you need to submit your project while we are still accepting submissions.**\n\nYou can start learning and working through the materials right away. For starting, you can refer to the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/) and the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for the workflow.', additional_kwargs={}, response_metadata={'model': 'batiia-gemma-mini:latest', 'created_at': '2026-06-18T06:08:44.07979887Z', 'done': True, 'done_reason': 'stop', 'total_duration': 32844866967, 'load_duration': 11750111497, 'prompt_eval_count': 865, 'prompt_eval_duration': 5419117550, 'eval_count': 427, 'eval_duration': 14996536639, 'logprobs': None, 'model_name': 'batiia-gemma-mini:latest', 'model_provider': 'ollama'}, id='lc_run--019ed958-2866-7fe0-b938-11be4f5df850-0', tool_calls=[], invalid_too